# Pilot 4 · Behavioural follow-up

Saved answers → verified source bundle → harmonized scoring report.

**CPU only.** This release implements source/scoring, paired transitions, sampled reliability and the intended-answer audit (Tickets 01–04). Human labels are supplied by the reviewer. No training or new generation runs here.

**Local release:** the pinned source has not been published to GitHub. Run locally, or supply the exact checkout in Colab; this is not a ready public Colab link.

**Resume:** run 1–2, then 4 to continue scoring, 5 for score tables, 6 for transitions or 7 for sampled reliability. Sections 8–12 cover the audit; run setup and settings before the desired section. Source selection is needed only when importing a new bundle.

See `docs/pilot4-behaviour-breakdown.md` for the manifest contract and source-evidence limits.


## 1. Load the CPU harness

Use the repository checkout and existing Python/NumPy environment. Colab also mounts Drive. No Torch, Transformers or CUDA setup is needed.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    drive.mount('/content/drive')

HARNESS_COMMIT = '508fee1ea1e3c64ccd1e3db69fd57e8e6d43600e'
if IN_COLAB:
    # Supply a checkout of this local release; it is not published to GitHub yet.
    REPO_DIR = Path(os.environ.get('SPAR_REPO_DIR', '/content/pilot4-followup-harness'))
    if not (REPO_DIR / 'pilot_eval').is_dir():
        raise RuntimeError('Upload the local release and set SPAR_REPO_DIR, or publish the pinned release before using Colab.')
    actual = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
    if actual != HARNESS_COMMIT:
        raise RuntimeError('Checkout differs from the notebook source pin; select the pinned release.')
else:
    REPO_DIR = Path(os.environ.get('SPAR_REPO_DIR', Path.cwd())).resolve()
    if not (REPO_DIR / 'pilot_eval').is_dir():
        raise RuntimeError('Start Jupyter in the repository root or set SPAR_REPO_DIR.')

loaded = sys.modules.get('pilot_eval')
if loaded and not Path(loaded.__file__).resolve().is_relative_to(REPO_DIR):
    raise RuntimeError('Restart the kernel: another harness checkout is already imported.')
sys.path.insert(0, str(REPO_DIR))
from pilot_eval.notebook_session import NotebookSession

print('CPU saved-output workflow. No model libraries or GPU are required.')


## 2. Select the analysis and restore its saved paths

A new name creates a separate immutable analysis. Reuse the same name to resume verified work.


In [ ]:
ANALYSIS_NAME = 'pilot4-behaviour-breakdown-v1'  # @param {type:"string"}
SOURCE_SELECTION_PATH = ''  # @param {type:"string"}

ANALYSIS_NAME = os.environ.get('SPAR_BEHAVIOUR_NAME', ANALYSIS_NAME)
default_root = Path('/content/drive/MyDrive/SPAR/spar-pilot/artifacts') if IN_COLAB else REPO_DIR / 'artifacts'
ARTIFACT_ROOT = Path(os.environ.get('SPAR_BEHAVIOUR_ROOT', default_root)).resolve()
runner = NotebookSession(ARTIFACT_ROOT, REPO_DIR, plan_name=ANALYSIS_NAME)
saved = NotebookSession.saved_selection(ARTIFACT_ROOT, ANALYSIS_NAME)
SOURCE_SELECTION_PATH = os.environ.get('SPAR_BEHAVIOUR_SELECTION', SOURCE_SELECTION_PATH or saved.get('behaviour_selection', ''))
CONFIG_PATH = runner.plan(ANALYSIS_NAME, 'behaviour.analysis.json')
print('Saved plan:', CONFIG_PATH, '| exists:', CONFIG_PATH.is_file())

AUDIT_NAME = ''  # @param {type:"string"}
AUDIT_NAME = os.environ.get('SPAR_AUDIT_NAME', AUDIT_NAME or saved.get('audit_name', ANALYSIS_NAME + '-audit-v1'))
AUDIT_CONFIG = runner.plan(AUDIT_NAME, 'behaviour.audit.json')


## 3. Verify and import explicit source exports

Reads the selected files, checks identities and recognized stored scores, and copies a source-bound bundle. Existing evidence remains unchanged. Missing provenance is reported rather than inferred.


In [ ]:
if SOURCE_SELECTION_PATH and Path(SOURCE_SELECTION_PATH).is_file():
    runner.run([sys.executable, '-u', '-m', 'pilot_eval', 'behaviour-prepare',
                '--selection', SOURCE_SELECTION_PATH, '--name', ANALYSIS_NAME,
                '--output-root', str(ARTIFACT_ROOT)])
    runner.save_selection({'behaviour_selection': SOURCE_SELECTION_PATH})
elif CONFIG_PATH.is_file():
    print('Restored existing plan; scoring/verification will check its saved inputs.')
else:
    raise ValueError('Set SOURCE_SELECTION_PATH to the explicit source manifest described in the workflow guide.')


## 4. Score or resume endpoints

Produces strict, flexible-v3 and last-number decisions. Completed sealed units are reused; altered inputs or outputs stop the command.


In [ ]:
CONFIG_PATH = runner.plan(ANALYSIS_NAME, 'behaviour.analysis.json')
runner.run([sys.executable, '-u', '-m', 'pilot_eval', 'behaviour-score',
            '--config', str(CONFIG_PATH), '--output-root', str(ARTIFACT_ROOT)])


## 5. Verify and read the scoring report

Shows original denominators, extractability, historical-score evidence and source compatibility. This is not yet the transition or human-audit report.


In [ ]:
CONFIG_PATH = runner.plan(ANALYSIS_NAME, 'behaviour.analysis.json')
runner.run([sys.executable, '-u', '-m', 'pilot_eval', 'behaviour-verify',
            '--config', str(CONFIG_PATH), '--output-root', str(ARTIFACT_ROOT)])
config = json.loads(CONFIG_PATH.read_text())
run_directory = runner.artifact(config['run_path'])
report_text = (run_directory / 'results/report.md').read_text()
print(report_text)
print('Structured results:', run_directory / 'results/results.json')
print('Raw and harmonized records:', run_directory / 'results/responses.jsonl')


## 6. Paired greedy transitions (CPU)

Uses the verified scoring bundle. Saves correct/wrong/invalid flows, gains and losses, paired intervals, direct GRPO64–SFT64 comparison and SFT length descriptions. Re-running verifies and restores completed outputs.


In [ ]:
CONFIG_PATH = runner.plan(ANALYSIS_NAME, 'behaviour.analysis.json')
TRANSITION_NAME = ANALYSIS_NAME + '-transitions-v1'
runner.run([sys.executable, '-u', '-m', 'pilot_eval', 'behaviour-transitions',
            '--config', str(CONFIG_PATH), '--name', TRANSITION_NAME,
            '--output-root', str(ARTIFACT_ROOT)])
transition_directory = runner.artifact('reports/' + TRANSITION_NAME)
print((transition_directory / 'results/report.md').read_text())
print('Per-question evidence:', transition_directory / 'results/paired-items.jsonl')
print('Plot:', transition_directory / 'plots/accuracy-differences.svg')


## 7. Observed eight-draw GRPO reliability (CPU)

Choose the frozen native GRPO source index from the inventory below. The source must include all eight saved held-out draws at steps 0 and 64. This measures observed coverage and mean reward; it does not generate responses or establish new ability. The choice is saved for the next session.


In [ ]:
SAMPLED_SOURCE_INDEX = -1  # @param {type:"integer"}
CONFIG_PATH = runner.plan(ANALYSIS_NAME, 'behaviour.analysis.json')
source_plan = json.loads(CONFIG_PATH.read_text())
for index, source in enumerate(source_plan['sources']):
    print(index, '|', source['method'], '|', source['format'])
saved_sampling = NotebookSession.saved_selection(ARTIFACT_ROOT, ANALYSIS_NAME)
selected_index = int(os.environ.get('SPAR_SAMPLED_SOURCE_INDEX',
    saved_sampling.get('sampled_source_index', SAMPLED_SOURCE_INDEX) if SAMPLED_SOURCE_INDEX < 0 else SAMPLED_SOURCE_INDEX))
if selected_index < 0:
    raise ValueError('Set SAMPLED_SOURCE_INDEX to the native GRPO source shown above.')
SAMPLING_NAME = ANALYSIS_NAME + '-sampling-v1'
runner.run([sys.executable, '-u', '-m', 'pilot_eval', 'behaviour-sampling',
            '--config', str(CONFIG_PATH), '--sampled-source-index', str(selected_index),
            '--name', SAMPLING_NAME, '--output-root', str(ARTIFACT_ROOT)])
runner.save_selection({'sampled_source_index': selected_index})
sampling_directory = runner.artifact('reports/' + SAMPLING_NAME)
print((sampling_directory / 'results/report.md').read_text())
print('Per-question counts:', sampling_directory / 'results/per-item.jsonl')
print('Plot:', sampling_directory / 'plots/observed-coverage.svg')


## 8. Prepare or restore the blinded audit (CPU)

Select the completed transition report. This saves the agreed mandatory cases and controls, prints the workload and exports an offline review page. Only share the `reviewer` folder with the reviewer; the private mapping stays separate. Browser caching is temporary—download drafts and save them in section 9.


In [ ]:
TRANSITIONS_REPORT_PATH = ''  # @param {type:"string"}
DOWNLOAD_REVIEW_PAGE = False  # @param {type:"boolean"}
AUDIT_CONFIG = runner.plan(AUDIT_NAME, 'behaviour.audit.json')
saved_audit = NotebookSession.saved_selection(ARTIFACT_ROOT, ANALYSIS_NAME)
if AUDIT_CONFIG.is_file():
    bound_report = json.loads(AUDIT_CONFIG.read_text())['transitions_path']
else:
    bound_report = saved_audit.get('audit_transitions', 'reports/' + ANALYSIS_NAME + '-transitions-v1')
transition_path = runner.artifact(os.environ.get('SPAR_AUDIT_TRANSITIONS', TRANSITIONS_REPORT_PATH or bound_report))
runner.run([sys.executable, '-u', '-m', 'pilot_eval', 'behaviour-audit-prepare',
            '--transitions', str(transition_path), '--name', AUDIT_NAME,
            '--output-root', str(ARTIFACT_ROOT)])
runner.save_selection({'audit_name': AUDIT_NAME, 'audit_transitions': str(transition_path)})
runner.run([sys.executable, '-u', '-m', 'pilot_eval', 'behaviour-audit-status',
            '--config', str(AUDIT_CONFIG), '--output-root', str(ARTIFACT_ROOT)])
audit_plan = json.loads(AUDIT_CONFIG.read_text())
review_page = runner.artifact(audit_plan['run_path']) / 'reviewer/review.html'
print('Open the offline reviewer page:', review_page)
if IN_COLAB and DOWNLOAD_REVIEW_PAGE:
    from google.colab import files
    files.download(str(review_page))


## 9. Save a downloaded answer draft (CPU)

Enter the downloaded `answers.json` path. Saves a sealed revision and regenerates `reviewer/draft.json`; import that updated draft into the page before continuing. Repeating the same import is safe. Unknown IDs, a different packet, malformed numbers or a stale revision stop the command.


In [ ]:
ANSWER_DRAFT_PATH = ''  # @param {type:"string"}
AUDIT_CONFIG = runner.plan(AUDIT_NAME, 'behaviour.audit.json')
answer_draft = os.environ.get('SPAR_AUDIT_LABELS', ANSWER_DRAFT_PATH)
if not answer_draft or not Path(answer_draft).is_file():
    raise ValueError('Set ANSWER_DRAFT_PATH to the downloaded answers.json file.')
runner.run([sys.executable, '-u', '-m', 'pilot_eval', 'behaviour-audit-save',
            '--config', str(AUDIT_CONFIG), '--labels', str(answer_draft),
            '--output-root', str(ARTIFACT_ROOT)])
runner.run([sys.executable, '-u', '-m', 'pilot_eval', 'behaviour-audit-status',
            '--config', str(AUDIT_CONFIG), '--output-root', str(ARTIFACT_ROOT)])
print('Import the updated reviewer/draft.json shown above before continuing the page draft.')


## 10. Freeze complete intended-answer labels (CPU)

Only run after the reviewer has completed every response. Check the box explicitly. Pending labels block freezing. Frozen labels are immutable; later corrections are recorded as exposed revisions.


In [ ]:
FREEZE_REVIEWED_LABELS = False  # @param {type:"boolean"}
if not (FREEZE_REVIEWED_LABELS or os.environ.get('SPAR_AUDIT_FREEZE') == '1'):
    raise ValueError('Review all intended-answer labels, then explicitly enable FREEZE_REVIEWED_LABELS.')
AUDIT_CONFIG = runner.plan(AUDIT_NAME, 'behaviour.audit.json')
runner.run([sys.executable, '-u', '-m', 'pilot_eval', 'behaviour-audit-freeze',
            '--config', str(AUDIT_CONFIG), '--output-root', str(ARTIFACT_ROOT)])


## 11. Reveal answer and scorer comparisons (CPU)

Requires the frozen labels. Produces the selected-case report and a separate explanation page, now showing gold and source/scorer information. It does not estimate whole-cohort human accuracy or reasoning validity.


In [ ]:
AUDIT_CONFIG = runner.plan(AUDIT_NAME, 'behaviour.audit.json')
runner.run([sys.executable, '-u', '-m', 'pilot_eval', 'behaviour-audit-reveal',
            '--config', str(AUDIT_CONFIG), '--output-root', str(ARTIFACT_ROOT)])
audit_plan = json.loads(AUDIT_CONFIG.read_text())
reveal_directory = runner.artifact(audit_plan['run_path']) / 'reveal'
print('Selected-case report:', reveal_directory / 'report.md')
print('Explanation page:', reveal_directory / 'review.html')


## 12. Save explanatory annotations (CPU)

The explanation page downloads `explanations.json`. Save each snapshot under a new revision name. Partial annotation remains explicit; these explanations never overwrite intended-answer labels. The guide documents separately named post-reveal answer corrections.


In [ ]:
EXPLANATION_DRAFT_PATH = ''  # @param {type:"string"}
EXPLANATION_REVISION = 'review-v1'  # @param {type:"string"}
EXPLANATION_SOURCE_REVISION = ''  # @param {type:"string"}
AUDIT_CONFIG = runner.plan(AUDIT_NAME, 'behaviour.audit.json')
if not EXPLANATION_DRAFT_PATH or not Path(EXPLANATION_DRAFT_PATH).is_file():
    raise ValueError('Set EXPLANATION_DRAFT_PATH to the downloaded explanations.json file.')
command = [sys.executable, '-u', '-m', 'pilot_eval', 'behaviour-audit-explain',
           '--config', str(AUDIT_CONFIG), '--annotations', EXPLANATION_DRAFT_PATH,
           '--revision', EXPLANATION_REVISION, '--output-root', str(ARTIFACT_ROOT)]
if EXPLANATION_SOURCE_REVISION.strip():
    command += ['--source-revision', EXPLANATION_SOURCE_REVISION]
runner.run(command)
